In [1]:
import pandas as pd
df = pd.read_csv("./EDA.csv")
df.head()

,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,...,quarter,epoka,total_actors_popularity,total_crew_popularity,main_genre,one_or_more_origin,origin_in_US,origin_in_top_5,keywords_list,n_keywords
0,5.0,Four Rooms,4000000.0,4257354,1995-12-09,98,en,3.8441,5.892,2810,...,4,1990-2009,21.0431,12.9056,comedy,1.0,1,0,"['hotel', 'storytelling', 'tarantino', 'multip...",5
1,6.0,Judgment Night,21000000.0,12136938,1993-10-15,109,en,2.2315,6.469,368,...,4,1990-2009,14.2132,3.8990,action,1.0,1,0,"['gangsters', 'goodaction', 'chicago', 'chase']",4
2,11.0,Star Wars,11000000.0,775398007,1977-05-25,121,en,20.6912,8.200,22061,...,2,1970-1989,19.0900,6.8303,adventure,1.0,1,0,"['totalitarianism', 'hermit', 'smuggling(contr...",22
3,12.0,Finding Nemo,94000000.0,940335536,2003-05-30,100,en,16.5979,7.817,20364,...,2,1990-2009,20.5784,4.2884,animation,1.0,1,0,"['short-termmemoryloss', 'pixaranimation', 'os...",4
4,13.0,Forrest Gump,55000000.0,677387716,1994-06-23,142,en,26.0309,8.500,29387,...,2,1990-2009,53.1438,5.2208,comedy,1.0,1,0,"['oscar', 'imdbtop250']",2


In [2]:
df['keywords_list']

,keywords_list
0,"['hotel', 'storytelling', 'tarantino', 'multip..."
1,"['gangsters', 'goodaction', 'chicago', 'chase']"
2,"['totalitarianism', 'hermit', 'smuggling(contr..."
3,"['short-termmemoryloss', 'pixaranimation', 'os..."
4,"['oscar', 'imdbtop250']"
...,...
19115,"['historicalfiction', 'silkroad', 'timetravel'..."
19116,"['surfer', 'surfing', 'indonesia', 'ripcurl', ..."
19117,['religiousfilm']
19118,"['suspenseful', 'tokusatsu', 'shocking']"


In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Generowanie wektorów... To może chwilę potrwać przy dużej ilości danych.")
embeddings = model.encode(df['keywords_list'], show_progress_bar=True)
print(f"Kształt macierzy wynikowej: {embeddings.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generowanie wektorów... To może chwilę potrwać przy dużej ilości danych.


Batches:   0%|          | 0/598 [00:00<?, ?it/s]

Kształt macierzy wynikowej: (19120, 384)


In [4]:
embeddings

array([[ 0.29802343, -0.5899861 , -0.005342  , ...,  0.07866912,
         0.05829569, -0.18205726],
       [-0.12741037, -0.16047114, -0.10513673, ...,  0.02937806,
         0.5046119 , -0.01054183],
       [ 0.07076234,  0.00434099, -0.09433934, ...,  0.22868699,
         0.2717812 , -0.10548694],
       ...,
       [ 0.22424382,  0.29025596, -0.40317938, ..., -0.39669186,
         0.28956133,  0.04008575],
       [ 0.1768477 , -0.13411076,  0.10183746, ...,  0.3889097 ,
         0.09930947,  0.08384632],
       [ 0.00195734, -0.10533068,  0.11586512, ...,  0.16228563,
         0.5613628 ,  0.05690777]], dtype=float32)

In [17]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# 1. Redukcja wymiarów (opcjonalna, np. do 50, żeby przyspieszyć KMeans)
pca = PCA(n_components=50)
embeddings_reduced = pca.fit_transform(embeddings)

# 2. Definiujemy liczbę klastrów (np. 10 grup tematycznych)
num_clusters = 50
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)

# 3. Trenowanie i przypisanie do klastrów
df['cluster'] = kmeans.fit_predict(embeddings_reduced)

# 4. Sprawdzenie wyników - co jest w klastrze 0?
print(df[df['cluster'] == 0]['keywords'].head(10))

6              oscar,goldenpalm,heartbreaking,music,dark
24            hiphop,ragstoriches,musicians,music,biopic
28             biopic,biographical,oscar,musicians,music
77     hellsangels,concertfilm,rockconcert,directcine...
93              caper,heist,cool,conartists,ensemblecast
152    relationships,music,interracialromance,awesome...
174    opera,composer,musician,marriagecrisis,italy,t...
183         hiphop,music,musicians,documentary,goodmusic
266    friendship,childabuse,rock'n'roll,musicrecord,...
309    prison,chicago,illinois,dancing,concert,countr...
Name: keywords, dtype: object


In [18]:
print(df[df['cluster'] == 1]['keywords'].head(10))

52             oscar,almodovar,spanish,women,transgender
185    farm,comingout,casino,basedonnovelorbook,nevad...
304         indians,women,lesbian,india,socialcommentary
781    prostitute,paris,france,philosophy,pimp,female...
885    paris,france,marriagecrisis,women'ssexualident...
921    daughter,berlin,germany,women'sfootball(soccer...
922    prostitute,telaviv,israel,escortservice,teache...
925                workplace,hospital,drama,sex,original
934    siblingrelationship,paris,france,marriagecrisi...
951           women,secrets,murdermystery,french,musical
Name: keywords, dtype: object


In [19]:
for i in range(num_clusters):
    cluster_keywords = " ".join(df[df['cluster'] == i]['keywords'].astype(str)).split()
    # Tutaj możesz użyć Counter, żeby wyciągnąć TOP 5 słów dla każdego klastra
    from collections import Counter
    most_common = Counter(cluster_keywords).most_common(5)
    print(f"Klaster {i}: {most_common}")

Klaster 0: [('concert', 4), ('popmusic', 2), ('rockandroll,musicians,music,documentary,intimate', 2), ('musicdocumentary', 2), ('compilation,rockband,girlband,editedfromtvseries,seinen,anime,socialanxiety,recap,music', 2)]
Klaster 1: [('prostitution', 2), ('femalefriendship', 2), ('paris,france,womandirector', 2), ('oscar,almodovar,spanish,women,transgender', 1), ('farm,comingout,casino,basedonnovelorbook,nevada,reno,nevada,lesbianrelationship,divorce,desert,lgbt,womandirector,1950s,englishprofessor,lesbian', 1)]
Klaster 2: [('nan', 2650), ('invisibleperson', 1)]
Klaster 3: [('drugs', 3), ('drugabuse,drugaddiction,addiction,drugs,heroin', 2), ('drugs,latinamerica,newyork,heroin,drugaddiction', 1), ('drugabuse,supportgroup,1970s,drugaddiction,junkie,portland,oregon,drugstore,grouptherapy,melancholy,drugstealing,burglars,opioidusedisorder,hopeful,tragic', 1), ('violence,heist,bankrobbery,drugs,violent', 1)]
Klaster 4: [('holocaust,jews,worldwarii,nazis,nazi', 3), ('genocide,truestory,civ

In [12]:
(print(len(df[df['cluster'] == 1]['keywords']
           )))

2131


In [15]:
for i in range(12):
  (print(len(df[df['cluster'] == i]['keywords'])))

1317
2131
2651
1399
2439
2767
2266
1490
338
2322
0
0


In [20]:
df.head()

,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,...,epoka,total_actors_popularity,total_crew_popularity,main_genre,one_or_more_origin,origin_in_US,origin_in_top_5,keywords_list,n_keywords,cluster
0,5.0,Four Rooms,4000000.0,4257354,1995-12-09,98,en,3.8441,5.892,2810,...,1990-2009,21.0431,12.9056,comedy,1.0,1,0,"['hotel', 'storytelling', 'tarantino', 'multip...",5,33
1,6.0,Judgment Night,21000000.0,12136938,1993-10-15,109,en,2.2315,6.469,368,...,1990-2009,14.2132,3.8990,action,1.0,1,0,"['gangsters', 'goodaction', 'chicago', 'chase']",4,30
2,11.0,Star Wars,11000000.0,775398007,1977-05-25,121,en,20.6912,8.200,22061,...,1970-1989,19.0900,6.8303,adventure,1.0,1,0,"['totalitarianism', 'hermit', 'smuggling(contr...",22,44
3,12.0,Finding Nemo,94000000.0,940335536,2003-05-30,100,en,16.5979,7.817,20364,...,1990-2009,20.5784,4.2884,animation,1.0,1,0,"['short-termmemoryloss', 'pixaranimation', 'os...",4,32
4,13.0,Forrest Gump,55000000.0,677387716,1994-06-23,142,en,26.0309,8.500,29387,...,1990-2009,53.1438,5.2208,comedy,1.0,1,0,"['oscar', 'imdbtop250']",2,13


In [22]:
df.to_csv("./pre_clean_data.csv",index=False)